V(s)=How good is being in state s

V(s)=E a∼π [Q(s,a)]

average over all possible actions (weighted by policy)

🔹 3. How is entropy used in A2C?

We add it to the loss:

Loss=policy loss+c⋅value loss−β⋅entropy

In [1]:
#imports

import numpy as np 
import torch
import torch.nn as nn 
from torch import optim 
from tqdm import tqdm 
import gymnasium as gym 

In [14]:
class A2C_agent():
    def __init__(
            self,
            n_env, 
            critic_lr,
            actor_lr,
            gamma,
            lam,
            ent_coef,
            env_name = "LunarLander-v3",
            is_randomize = 1,
            max_episode_steps = 600,
            seed = 42
    ):


        self.critic_lr = critic_lr
        self.actor_lr = actor_lr
        self.gamma = gamma
        self.lam = lam
        self.seed = seed
        self.ent_coef = ent_coef
        self.n_envs = n_env
        self.actor_weights_path = "weights/actor_weights.pth"
        self.critic_weights_path = "weights/critic_weights.pth"
        self.env_name = env_name

        self.max_episode_steps = max_episode_steps

        self.device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
        if(is_randomize):
            self.envs = gym.vector.SyncVectorEnv(
    [
        lambda : gym.make(env_name,
                          gravity = np.clip(np.random.normal(loc=-10.0 , scale=1.0), a_min = -11.99 , a_max = -0.01),
                          enable_wind = np.random.choice([True , False]),
                          wind_power = np.clip(np.random.normal(loc=15.0 , scale=1.0), a_min=0.01 , a_max=19.99),
                          turbulence_power= np.clip(np.random.normal(loc=1.5, scale=0.5), a_min=0.01 , a_max= 1.99),
                          max_episode_steps = self.max_episode_steps,
                          ) for i in range (self.n_envs)               
    ])
        else:
            self.envs = gym.make_vec(env_name , num_envs=self.n_envs , max_episode_steps = self.max_episode_steps)

        self.n_actions = self.envs.single_action_space.n 
        self.n_states = self.envs.single_observation_space.shape[0]

    def create_A2C_net(self):
        class A2C_net(nn.Module):
            def __init__(self, n_actions, n_states):
                
                super ().__init__()


                hidden_layer_1 = 32
                hidden_layer_2 = 32

                self.critic = nn.Sequential(
                    nn.Linear(n_states,hidden_layer_1),
                    nn.ReLU(),
                    nn.Linear(hidden_layer_1, hidden_layer_2),
                    nn.ReLU(),
                    nn.Linear(hidden_layer_2,1),
                )

                self.actor = nn.Sequential(
                    nn.Linear(n_states,hidden_layer_1),
                    nn.ReLU(),
                    nn.Linear(hidden_layer_1, hidden_layer_2),
                    nn.ReLU(),
                    nn.Linear(hidden_layer_2,n_actions),
                )

            def forward(self , state):
                state_values = self.critic(state)
                action_logits_vec = self.actor(state)
                return (state_values, action_logits_vec)
            
        self.A2C = A2C_net(self.n_actions , self.n_states).to(self.device)

        self.crit_optim = optim.RMSprop(self.A2C.critic.parameters(), lr = self.critic_lr)
        self.actor_optim = optim.RMSprop(self.A2C.actor.parameters(), lr = self.actor_lr)

    def get_actions (self, states):
        states = torch.Tensor(states).to(self.device)
        state_values,action_logits = self.A2C.forward(states)
        action_pd = torch.distributions.Categorical(logits = action_logits)
        actions = action_pd.sample()
        action_log_probs = action_pd.log_prob(actions)
        entropy = action_pd.entropy()
        return actions , action_log_probs , state_values , entropy
    
    def get_action (self, states):
        with torch.no_grad():
            action, _, _, _ = self.get_actions(states[None, :])

        return action.item()
    def visual_evaluate(self , n_itr):
        env = gym.make(self.env_name , render_mode = "human")
        for itr in range(n_itr) :
            done = False
            state , info = env.reset()
            while not done:
                
                action = self.get_action(state)
                state, reward, terminated, truncated, info = env.step(action)
                done = terminated or truncated
            
            
        env.close()
    
    def compute_loss(
            self,
            rewards ,
            action_log_probs ,
            value_preds,
            entropy,
            masks,
            gamma,
            lam,
            ent_coef
    ):
        T = len(rewards)
        advantages = torch.zeros(T , self.n_envs).to(self.device)
        gae = 0   #GAE (Generalized Advantage Estimation)
        
        for t in reversed(range(T-1)):
            td_error = (
                rewards[t] + gamma*masks[t]*value_preds[t+1] - value_preds[t]
            )
            gae = td_error + gamma * lam * masks[t] *gae 
            advantages[t] = gae
        
        critic_loss = advantages.pow(2).mean()
        actor_loss = -(advantages.detach()*action_log_probs).mean() - ent_coef*entropy.mean()
        return critic_loss ,  actor_loss
    
    def update_params(
            self ,
            crit_loss , 
            actor_loss
    ):
        self.crit_optim.zero_grad()
        crit_loss.backward()
        self.crit_optim.step()

        self.actor_optim.zero_grad()
        actor_loss.backward()
        self.actor_optim.step()

    def train(self , n_episodes , n_steps_per_update):

        for episode in tqdm(range(n_episodes)):
            ep_value_preds = torch.zeros(n_steps_per_update, self.n_envs, device=self.device)
            ep_rewards = torch.zeros(n_steps_per_update , self.n_envs , device=self.device)
            ep_action_log_probs = torch.zeros(n_steps_per_update,self.n_envs , device=self.device)
            ep_entropies = torch.zeros(n_steps_per_update,self.n_envs , device=self.device)
            masks = torch.zeros(n_steps_per_update,self.n_envs ,device=self.device)

            if episode == 0:
                states , infos = self.envs.reset(seed = self.seed)
            for step in range(n_steps_per_update):
                actions , action_log_probs , state_value_preds , entropy = self.get_actions(states)
                states , rewards , terminated , truncated , infos = self.envs.step(actions.cpu().numpy())

                ep_value_preds[step] = torch.squeeze(state_value_preds)
                ep_rewards[step] = torch.tensor(rewards, device=self.device)
                ep_entropies[step] = entropy
                ep_action_log_probs[step] = action_log_probs
                masks[step] = torch.tensor([not (t or tr) for t, tr in zip(terminated, truncated)])
            critic_loss , actor_loss = self.compute_loss(
                ep_rewards,
                ep_action_log_probs,
                ep_value_preds,
                ep_entropies,
                masks,
                self.gamma,
                self.lam,
                self.ent_coef
            )

            self.update_params(critic_loss , actor_loss)


    def save_agent(self):
        torch.save(self.A2C.actor.state_dict(), self.actor_weights_path)
        torch.save(self.A2C.critic.state_dict(), self.critic_weights_path)

    def load_agent(self):
            self.A2C.actor.load_state_dict(torch.load(self.actor_weights_path))
            self.A2C.critic.load_state_dict(torch.load(self.critic_weights_path))
            self.A2C.actor.eval()
            self.A2C.critic.eval()

        


In [15]:
n_envs = 10 
n_update = 1000
n_steps_per_update = 128
randomize_domain = False

gamma = 0.999
lam = 0.95 
ent_coef = 0.01 
actor_lr = 0.001
critic_lr = 0.005

            # n_env, 
            # critic_lr,
            # actor_lr,
            # gamma,
            # lam,
            # ent_coef,
            # env_name = "LunarLander-v3",
            # is_randomize = 1,
            # max_episode_steps = 600,
            # seed = 42

agent = A2C_agent(
    n_envs , critic_lr,actor_lr,gamma , lam , ent_coef
)

In [16]:
agent.create_A2C_net()

In [11]:
n_episodes = 1000
n_steps_per_update = 128

agent.train(n_episodes,n_steps_per_update)

  1%|          | 6/1000 [00:03<10:14,  1.62it/s]


KeyboardInterrupt: 

In [17]:
agent.load_agent()

C:\Users\User\AppData\Local\Temp\ipykernel_11864\2162015009.py:190: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.A2C.actor.load_state_dict(torch.load(self.actor_weight

In [18]:
agent.visual_evaluate(10)

KeyboardInterrupt: 